# 09 · Two ways to reward code: similarity vs execution

The difference between a **rule-based similarity reward**
(SWE-RL style) and an **execution reward**, and why the choice matters.

**Key concepts**
- **Rule-based / similarity reward (SWE-RL)** — score a candidate by its textual
  similarity to a ground-truth solution (`difflib`). Cheap, no execution, but blind
  to behavior — a correct-but-different solution scores low.
- **Execution reward** — run the candidate against tests in a sandbox; rewards
  *behavioral* correctness regardless of wording. More faithful, needs an executable
  environment.
- **Agreement vs divergence** — comparing both over several samples shows when a
  cheap proxy is misleading (the key design lesson for RL reward functions).

**How it works:** sample candidates from Claude (Bedrock), compute both rewards, and
tabulate where they agree or diverge.

**Prerequisites:** AWS credentials + Bedrock model access.

In [ ]:
%pip install -q raitbox

### Configuration & candidates
Bedrock client, the ground-truth `ORACLE` + `TESTS`, and `sample` to draw candidate solutions.

In [ ]:
import difflib
import os
import re

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

# Ground-truth solution + its tests.
ORACLE = (
    "def two_sum(nums, target):\n"
    "    seen = {}\n"
    "    for i, n in enumerate(nums):\n"
    "        if target - n in seen:\n"
    "            return [seen[target - n], i]\n"
    "        seen[n] = i\n"
    "    return []\n"
)
TESTS = ("assert sorted(two_sum([2,7,11,15], 9)) == [0,1]; "
         "assert sorted(two_sum([3,2,4], 6)) == [1,2]; "
         "assert two_sum([1,2], 100) == []")
PROMPT = "Implement `def two_sum(nums, target)` returning indices of two numbers summing to target ([] if none)."


def sample(temperature: float) -> str:
    resp = bedrock.converse(modelId=MODEL_ID,
                            system=[{"text": "Reply with ONLY the function definition. No prose, no fences."}],
                            messages=[{"role": "user", "content": [{"text": PROMPT}]}],
                            inferenceConfig={"maxTokens": 600, "temperature": temperature})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (fenced[0] if fenced else text).strip()

### Two rewards, side by side
`similarity_reward` = `difflib` ratio to the oracle (SWE-RL — no execution); `execution_reward` = fraction of tests passing in the sandbox.

In [ ]:
def similarity_reward(candidate: str) -> float:
    # SWE-RL-style: sequence similarity to the ground truth (0..1).
    return round(difflib.SequenceMatcher(None, ORACLE, candidate).ratio(), 3)


def execution_reward(sbx, candidate: str) -> float:
    ex = sbx.run_code(candidate + "\n" + TESTS + "\nprint('OK')")
    return 1.0 if (ex.error is None and "OK" in "".join(ex.logs.stdout)) else 0.0

## Sample candidates and score both rewards

In [ ]:
candidates = [sample(t) for t in (0.0, 0.7, 0.7, 1.0)]

print(f"{'sim (SWE-RL)':>14} {'exec':>6}   verdict")
with Sandbox.create(region_name=REGION) as sbx:
    for i, cand in enumerate(candidates):
        sim = similarity_reward(cand)
        ex = execution_reward(sbx, cand)
        note = "agree" if (sim > 0.8) == (ex == 1.0) else "DIVERGE"
        print(f"{sim:>14.3f} {ex:>6.1f}   sample {i}: {note}")

print("\nTakeaway: similarity rewards textual closeness to one oracle; execution "
      "rewards *behavioral* correctness. They diverge for correct-but-different code.")

## Recap & where to go next
You contrasted a cheap proxy reward with a faithful executable reward and saw them diverge.

**Related:** `14`/`15` (execution reward in action) · `06` (test-driven repair).